In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

TRAIN_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/train")
TEST_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/test")
BASE_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")

# ── 1. FILE INVENTORY ─────────────────────────────────────────────────────────
train_hw = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_tw = sorted(TRAIN_DIR.glob("*__typewell.csv"))
test_hw  = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
test_tw  = sorted(TEST_DIR.glob("*__typewell.csv"))

train_ids = [f.stem.replace("__horizontal_well","") for f in train_hw]
test_ids  = [f.stem.replace("__horizontal_well","") for f in test_hw]

print("="*60)
print("FILE INVENTORY")
print("="*60)
print(f"Train wells : {len(train_ids)}")
print(f"Test  wells : {len(test_ids)}")
print(f"Train IDs sample : {train_ids[:5]}")
print(f"Test  IDs sample : {test_ids[:5]}")

# ── 2. SINGLE WELL DEEP DIVE ──────────────────────────────────────────────────
wid = train_ids[0]
hw  = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
tw  = pd.read_csv(TRAIN_DIR / f"{wid}__typewell.csv")

print("\n" + "="*60)
print(f"HORIZONTAL WELL SAMPLE: {wid}")
print("="*60)
print(f"Shape         : {hw.shape}")
print(f"Columns       : {hw.columns.tolist()}")
print(f"Dtypes:\n{hw.dtypes}")
print(f"\nFirst 3 rows:\n{hw.head(3).to_string()}")
print(f"\nMissing values:\n{hw.isna().sum()}")

print("\n" + "="*60)
print(f"TYPEWELL SAMPLE: {wid}")
print("="*60)
print(f"Shape         : {tw.shape}")
print(f"Columns       : {tw.columns.tolist()}")
print(f"Dtypes:\n{tw.dtypes}")
print(f"\nFirst 3 rows:\n{tw.head(3).to_string()}")
print(f"\nMissing values:\n{tw.isna().sum()}")
print(f"\nUnique Geology labels: {tw['Geology'].dropna().unique().tolist()}")

# ── 3. EVAL ZONE STRUCTURE ────────────────────────────────────────────────────
print("\n" + "="*60)
print("EVAL ZONE ANALYSIS (all train wells)")
print("="*60)

eval_stats = []
for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_nan  = hw["TVT_input"].isna()
    mask_known = hw["TVT_input"].notna()
    n_eval    = mask_nan.sum()
    n_known   = mask_known.sum()
    n_total   = len(hw)

    if n_eval == 0:
        eval_stats.append({"wid": wid, "n_total": n_total,
                           "n_eval": 0, "n_known": n_known,
                           "eval_pct": 0, "contiguous": True,
                           "eval_start_idx": -1, "eval_end_idx": -1,
                           "position": "none",
                           "context_before": False, "context_after": False})
        continue

    # Check if NaN block is contiguous
    nan_indices = np.where(mask_nan.values)[0]
    is_contiguous = (nan_indices[-1] - nan_indices[0] + 1) == len(nan_indices)

    eval_start = nan_indices[0]
    eval_end   = nan_indices[-1]

    # Context: known values before AND after eval zone?
    ctx_before = mask_known.values[:eval_start].any()
    ctx_after  = mask_known.values[eval_end+1:].any() if eval_end+1 < n_total else False

    if ctx_before and ctx_after:
        position = "middle"
    elif ctx_before and not ctx_after:
        position = "end"
    elif not ctx_before and ctx_after:
        position = "start"
    else:
        position = "all_nan"

    eval_stats.append({
        "wid": wid, "n_total": n_total,
        "n_eval": n_eval, "n_known": n_known,
        "eval_pct": round(n_eval/n_total*100, 1),
        "contiguous": is_contiguous,
        "eval_start_idx": eval_start, "eval_end_idx": eval_end,
        "position": position,
        "context_before": ctx_before, "context_after": ctx_after
    })

df_eval = pd.DataFrame(eval_stats)
print(f"Total train wells          : {len(df_eval)}")
print(f"Wells WITH eval zone       : {(df_eval.n_eval>0).sum()}")
print(f"Wells WITHOUT eval zone    : {(df_eval.n_eval==0).sum()}")
print(f"Contiguous NaN block (all) : {df_eval.contiguous.all()}")
print(f"\nEval zone POSITION counts:")
print(df_eval["position"].value_counts().to_string())
print(f"\nEval zone SIZE stats (rows):")
print(df_eval[df_eval.n_eval>0]["n_eval"].describe().round(1).to_string())
print(f"\nEval zone % of well stats:")
print(df_eval[df_eval.n_eval>0]["eval_pct"].describe().round(1).to_string())

# ── 4. TVT STATISTICS ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("TVT STATISTICS (train wells with eval zone)")
print("="*60)

tvt_stats = []
for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    if "TVT" not in hw.columns or hw["TVT"].isna().all():
        continue
    mask_nan = hw["TVT_input"].isna()
    tvt_all  = hw["TVT"].dropna()
    tvt_eval = hw.loc[mask_nan, "TVT"].dropna()
    tvt_known = hw.loc[~mask_nan, "TVT"].dropna()
    tvt_stats.append({
        "wid": wid,
        "tvt_min": tvt_all.min(), "tvt_max": tvt_all.max(),
        "tvt_range": tvt_all.max() - tvt_all.min(),
        "tvt_mean": tvt_all.mean(), "tvt_std": tvt_all.std(),
        "eval_tvt_min": tvt_eval.min() if len(tvt_eval)>0 else np.nan,
        "eval_tvt_max": tvt_eval.max() if len(tvt_eval)>0 else np.nan,
    })

df_tvt = pd.DataFrame(tvt_stats)
print(f"TVT global min  : {df_tvt.tvt_min.min():.2f}")
print(f"TVT global max  : {df_tvt.tvt_max.max():.2f}")
print(f"TVT range per well (mean): {df_tvt.tvt_range.mean():.2f}")
print(f"TVT range per well (std) : {df_tvt.tvt_range.std():.2f}")
print(f"TVT range per well (min) : {df_tvt.tvt_range.min():.2f}")
print(f"TVT range per well (max) : {df_tvt.tvt_range.max():.2f}")
print(f"\nTVT std per well (mean)  : {df_tvt.tvt_std.mean():.2f}")

# ── 5. FORMATION COLUMNS CHECK ────────────────────────────────────────────────
print("\n" + "="*60)
print("FORMATION COLUMNS AVAILABILITY")
print("="*60)
FORMATIONS = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']
form_avail = {f: 0 for f in FORMATIONS}
form_nan   = {f: [] for f in FORMATIONS}

for wid in train_ids[:50]:  # check first 50 wells
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    for f in FORMATIONS:
        if f in hw.columns:
            form_avail[f] += 1
            form_nan[f].append(hw[f].isna().mean())

for f in FORMATIONS:
    avg_nan = np.mean(form_nan[f]) if form_nan[f] else 1.0
    print(f"{f}: present in {form_avail[f]}/50 wells | avg NaN% = {avg_nan*100:.1f}%")

# ── 6. SAMPLE SUBMISSION ──────────────────────────────────────────────────────
print("\n" + "="*60)
print("SAMPLE SUBMISSION")
print("="*60)
sub = pd.read_csv(BASE_DIR / "sample_submission.csv")
print(f"Shape   : {sub.shape}")
print(f"Columns : {sub.columns.tolist()}")
print(f"Head:\n{sub.head(5).to_string()}")
print(f"ID format example: {sub['id'].iloc[0]}")
print(f"Total rows to predict: {len(sub)}")

FILE INVENTORY
Train wells : 773
Test  wells : 3
Train IDs sample : ['000d7d20', '00bbac68', '00e12e8b', '015fe0d2', '01869cd4']
Test  IDs sample : ['000d7d20', '00bbac68', '00e12e8b']

HORIZONTAL WELL SAMPLE: 000d7d20
Shape         : (5278, 13)
Columns       : ['MD', 'X', 'Y', 'Z', 'ANCC', 'ASTNU', 'ASTNL', 'EGFDU', 'EGFDL', 'BUDA', 'TVT', 'GR', 'TVT_input']
Dtypes:
MD           float64
X            float64
Y            float64
Z            float64
ANCC         float64
ASTNU        float64
ASTNL        float64
EGFDU        float64
EGFDL        float64
BUDA         float64
TVT          float64
GR           float64
TVT_input    float64
dtype: object

First 3 rows:
        MD           X           Y        Z     ANCC    ASTNU    ASTNL    EGFDU    EGFDL     BUDA       TVT          GR  TVT_input
0  11467.0  2983525.16  1069022.09 -9258.57 -9395.81 -9569.86 -9597.64 -9670.99 -9705.96 -9846.35  11236.02  115.692586   11236.02
1  11468.0  2983525.18  1069022.30 -9259.55 -9395.75 -9569.80 -959

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

TRAIN_DIR = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/train")
TEST_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction/test")

train_hw  = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_ids = [f.stem.replace("__horizontal_well","") for f in train_hw]
test_hw   = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
test_ids  = [f.stem.replace("__horizontal_well","") for f in test_hw]

FORMATIONS = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']

# ── 1. TVT STATISTICS ─────────────────────────────────────────────────────────
print("="*60)
print("TVT STATISTICS ACROSS ALL TRAIN WELLS")
print("="*60)

tvt_stats = []
for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_nan   = hw["TVT_input"].isna()
    mask_known = ~mask_nan
    tvt_all    = hw["TVT"]
    tvt_known  = hw.loc[mask_known, "TVT"]
    tvt_eval   = hw.loc[mask_nan,   "TVT"]

    tvt_stats.append({
        "wid"          : wid,
        "tvt_min"      : tvt_all.min(),
        "tvt_max"      : tvt_all.max(),
        "tvt_range"    : tvt_all.max() - tvt_all.min(),
        "tvt_std"      : tvt_all.std(),
        "last_known_tvt": tvt_known.iloc[-1] if len(tvt_known) > 0 else np.nan,
        "first_eval_tvt": tvt_eval.iloc[0]   if len(tvt_eval)  > 0 else np.nan,
        "last_eval_tvt" : tvt_eval.iloc[-1]  if len(tvt_eval)  > 0 else np.nan,
        "eval_tvt_range": tvt_eval.max() - tvt_eval.min() if len(tvt_eval) > 0 else np.nan,
        "n_known"      : mask_known.sum(),
        "n_eval"       : mask_nan.sum(),
    })

df = pd.DataFrame(tvt_stats)
print(f"TVT global min            : {df.tvt_min.min():.2f}")
print(f"TVT global max            : {df.tvt_max.max():.2f}")
print(f"TVT range/well  mean±std  : {df.tvt_range.mean():.1f} ± {df.tvt_range.std():.1f}")
print(f"TVT range/well  min/max   : {df.tvt_range.min():.1f} / {df.tvt_range.max():.1f}")
print(f"TVT std/well    mean      : {df.tvt_std.mean():.2f}")
print(f"\nEval TVT range/well mean  : {df.eval_tvt_range.mean():.1f}")
print(f"Eval TVT range/well max   : {df.eval_tvt_range.max():.1f}")

# Gap between last known and first eval
df["transition_jump"] = (df.first_eval_tvt - df.last_known_tvt).abs()
print(f"\nTransition jump at boundary mean : {df.transition_jump.mean():.4f}")
print(f"Transition jump at boundary max  : {df.transition_jump.max():.4f}")
print(f"(should be ~0 if TVT is continuous at boundary)")

# ── 2. GR MISSING PATTERN ────────────────────────────────────────────────────
print("\n" + "="*60)
print("GR MISSING PATTERN — INSIDE vs OUTSIDE EVAL ZONE")
print("="*60)

gr_stats = []
for wid in train_ids:
    hw         = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_nan   = hw["TVT_input"].isna()
    mask_known = ~mask_nan
    gr_miss_known = hw.loc[mask_known, "GR"].isna().mean()
    gr_miss_eval  = hw.loc[mask_nan,   "GR"].isna().mean()
    gr_stats.append({"gr_miss_known": gr_miss_known,
                     "gr_miss_eval" : gr_miss_eval})

gr_df = pd.DataFrame(gr_stats)
print(f"GR missing in KNOWN zone — mean: {gr_df.gr_miss_known.mean()*100:.1f}%")
print(f"GR missing in KNOWN zone — max : {gr_df.gr_miss_known.max()*100:.1f}%")
print(f"GR missing in EVAL  zone — mean: {gr_df.gr_miss_eval.mean()*100:.1f}%")
print(f"GR missing in EVAL  zone — max : {gr_df.gr_miss_eval.max()*100:.1f}%")
print(f"Wells with >50% GR missing in eval: {(gr_df.gr_miss_eval>0.5).sum()}")
print(f"Wells with 100% GR missing in eval: {(gr_df.gr_miss_eval==1.0).sum()}")

# ── 3. B_WELL BEHAVIOR (core physics check) ──────────────────────────────────
print("\n" + "="*60)
print("B_WELL BEHAVIOR: TVT + Z - formation_top")
print("(key physics: how stable is b_well in known zone?)")
print("="*60)

bwell_stats = []
for wid in train_ids:
    hw         = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_known = hw["TVT_input"].notna()
    kn         = hw[mask_known]
    if len(kn) < 10:
        continue
    for f in FORMATIONS:
        if f not in hw.columns:
            continue
        bwell = kn["TVT"].values + kn["Z"].values - kn[f].values
        bwell_stats.append({
            "formation" : f,
            "bwell_std" : float(np.std(bwell)),
            "bwell_mean": float(np.mean(bwell)),
            "bwell_drift": float(bwell[-1] - bwell[0]),  # drift over known zone
        })

bw_df = pd.DataFrame(bwell_stats)
print("Per formation — b_well stability in KNOWN zone:")
for f in FORMATIONS:
    sub = bw_df[bw_df.formation == f]
    print(f"  {f}: std mean={sub.bwell_std.mean():.2f}  "
          f"drift mean={sub.bwell_drift.mean():.2f}  "
          f"drift std={sub.bwell_drift.std():.2f}")

# ── 4. FORMATION COLUMNS NaN CHECK (all 773 wells) ───────────────────────────
print("\n" + "="*60)
print("FORMATION COLUMNS NaN CHECK (all 773 train wells)")
print("="*60)

for f in FORMATIONS:
    miss_wells = 0
    miss_pct   = []
    for wid in train_ids:
        hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
        if f not in hw.columns:
            miss_wells += 1
        else:
            miss_pct.append(hw[f].isna().mean())
    avg = np.mean(miss_pct)*100 if miss_pct else 100
    print(f"{f}: missing in {miss_wells} wells | avg NaN% in present wells = {avg:.1f}%")

# ── 5. TYPEWELL STATS ─────────────────────────────────────────────────────────
print("\n" + "="*60)
print("TYPEWELL STATISTICS")
print("="*60)

tw_lengths = []
tw_gr_ranges = []
for wid in train_ids:
    tw = pd.read_csv(TRAIN_DIR / f"{wid}__typewell.csv")
    tw_lengths.append(len(tw))
    tw_gr_ranges.append(tw["GR"].max() - tw["GR"].min())

print(f"Typewell length mean±std : {np.mean(tw_lengths):.0f} ± {np.std(tw_lengths):.0f}")
print(f"Typewell length min/max  : {np.min(tw_lengths)} / {np.max(tw_lengths)}")
print(f"Typewell GR range mean   : {np.mean(tw_gr_ranges):.1f}")
print(f"Typewell GR range min/max: {np.min(tw_gr_ranges):.1f} / {np.max(tw_gr_ranges):.1f}")

# ── 6. TEST WELL STRUCTURE ────────────────────────────────────────────────────
print("\n" + "="*60)
print("TEST WELL STRUCTURE")
print("="*60)

for wid in test_ids:
    hw = pd.read_csv(TEST_DIR / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TEST_DIR / f"{wid}__typewell.csv")
    mask_nan = hw["TVT_input"].isna()
    print(f"\nWell: {wid}")
    print(f"  HW shape       : {hw.shape}")
    print(f"  HW columns     : {hw.columns.tolist()}")
    print(f"  Eval zone rows : {mask_nan.sum()} / {len(hw)} "
          f"({mask_nan.mean()*100:.1f}%)")
    print(f"  HW has TVT col : {'TVT' in hw.columns}")
    print(f"  GR miss eval   : {hw.loc[mask_nan,'GR'].isna().mean()*100:.1f}%")
    print(f"  TW shape       : {tw.shape}")
    print(f"  TW columns     : {tw.columns.tolist()}")
    print(f"  TW has Geology : {'Geology' in tw.columns}")

TVT STATISTICS ACROSS ALL TRAIN WELLS
TVT global min            : 9245.19
TVT global max            : 12893.89
TVT range/well  mean±std  : 734.5 ± 175.5
TVT range/well  min/max   : 158.9 / 1256.7
TVT std/well    mean      : 136.62

Eval TVT range/well mean  : 29.4
Eval TVT range/well max   : 121.8

Transition jump at boundary mean : 0.0205
Transition jump at boundary max  : 0.5300
(should be ~0 if TVT is continuous at boundary)

GR MISSING PATTERN — INSIDE vs OUTSIDE EVAL ZONE
GR missing in KNOWN zone — mean: 23.6%
GR missing in KNOWN zone — max : 72.5%
GR missing in EVAL  zone — mean: 31.5%
GR missing in EVAL  zone — max : 81.2%
Wells with >50% GR missing in eval: 188
Wells with 100% GR missing in eval: 0

B_WELL BEHAVIOR: TVT + Z - formation_top
(key physics: how stable is b_well in known zone?)
Per formation — b_well stability in KNOWN zone:
  ANCC: std mean=0.01  drift mean=-0.00  drift std=0.01
  ASTNU: std mean=0.01  drift mean=-0.00  drift std=0.01
  ASTNL: std mean=0.01  drift 

In [3]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

BASE_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR  = BASE_DIR / "test"

train_hw  = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_ids = [f.stem.replace("__horizontal_well","") for f in train_hw]
test_hw   = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
test_ids  = [f.stem.replace("__horizontal_well","") for f in test_hw]

FORMATIONS = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']

# ── 1. VERIFY PHYSICS ON TRAIN EVAL ZONE ─────────────────────────────────────
# If TVT = -Z + formation_top + b_well is exact,
# then using known-zone b_well to predict eval TVT → RMSE should be near 0
print("="*60)
print("PHYSICS VERIFICATION: TVT = -Z + formation_top + b_well")
print("Applied to EVAL zone using b_well from KNOWN zone")
print("="*60)

rmse_by_form = {f: [] for f in FORMATIONS}
best_form_per_well = []

for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_known = hw["TVT_input"].notna()
    mask_eval  = hw["TVT_input"].isna()
    kn = hw[mask_known]
    ev = hw[mask_eval]
    if len(ev) == 0 or len(kn) < 5:
        continue

    well_rmses = {}
    for f in FORMATIONS:
        if f not in hw.columns:
            continue
        # b_well from known zone
        b_well = float(np.mean(kn["TVT"].values + kn["Z"].values - kn[f].values))
        # predict eval zone
        tvt_pred = -ev["Z"].values + ev[f].values + b_well
        tvt_true = ev["TVT"].values
        rmse = float(np.sqrt(np.mean((tvt_pred - tvt_true)**2)))
        rmse_by_form[f].append(rmse)
        well_rmses[f] = rmse

    best_f = min(well_rmses, key=well_rmses.get)
    best_form_per_well.append({"wid": wid, "best_form": best_f,
                                "best_rmse": well_rmses[best_f]})

print("Per-formation RMSE on EVAL zone (using known-zone b_well):")
for f in FORMATIONS:
    vals = rmse_by_form[f]
    print(f"  {f}: mean={np.mean(vals):.4f}  "
          f"std={np.std(vals):.4f}  "
          f"max={np.max(vals):.4f}  "
          f"p95={np.percentile(vals,95):.4f}")

bf_df = pd.DataFrame(best_form_per_well)
print(f"\nBest formation counts:")
print(bf_df["best_form"].value_counts().to_string())
print(f"\nBest formation RMSE stats:")
print(bf_df["best_rmse"].describe().round(4).to_string())
print(f"\nWells with best_rmse > 1.0  : {(bf_df.best_rmse > 1.0).sum()}")
print(f"Wells with best_rmse > 5.0  : {(bf_df.best_rmse > 5.0).sum()}")
print(f"Wells with best_rmse > 10.0 : {(bf_df.best_rmse > 10.0).sum()}")

# ── 2. SPATIAL STRUCTURE OF FORMATION TOPS ───────────────────────────────────
print("\n" + "="*60)
print("SPATIAL STRUCTURE OF FORMATION TOPS")
print("(median XY per well + median formation depth)")
print("="*60)

spatial = []
for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    row = {"wid": wid,
           "x_med": float(hw["X"].median()),
           "y_med": float(hw["Y"].median()),
           "z_med": float(hw["Z"].median())}
    for f in FORMATIONS:
        if f in hw.columns:
            row[f] = float(hw[f].median())
    spatial.append(row)

sp_df = pd.DataFrame(spatial)
print(f"Spatial X range: {sp_df.x_med.min():.0f} → {sp_df.x_med.max():.0f}  "
      f"span={sp_df.x_med.max()-sp_df.x_med.min():.0f} ft")
print(f"Spatial Y range: {sp_df.y_med.min():.0f} → {sp_df.y_med.max():.0f}  "
      f"span={sp_df.y_med.max()-sp_df.y_med.min():.0f} ft")
print(f"\nFormation top spatial variation (std across wells):")
for f in FORMATIONS:
    if f in sp_df.columns:
        print(f"  {f}: mean={sp_df[f].mean():.1f}  std={sp_df[f].std():.1f}  "
              f"range={sp_df[f].max()-sp_df[f].min():.1f}")

# ── 3. KNN FORMATION IMPUTATION TEST ─────────────────────────────────────────
# Simulate what we must do for test wells:
# For each train well, remove it and predict its formation tops
# using KNN from remaining train wells → measure imputation RMSE
print("\n" + "="*60)
print("KNN FORMATION IMPUTATION — LOO TEST")
print("(simulates test-well formation top prediction)")
print("="*60)

xy_all = sp_df[["x_med","y_med"]].values
scale  = xy_all.std(0); scale[scale<1] = 1
xy_norm = xy_all / scale

knn_rmse = {f: [] for f in FORMATIONS}
for k in [3, 5, 10, 20]:
    errs = {f: [] for f in FORMATIONS}
    tree = cKDTree(xy_norm)
    for i in range(len(sp_df)):
        dists, idx = tree.query(xy_norm[i], k=k+1)
        # exclude self
        idx_use = idx[idx != i][:k]
        w = 1.0 / (dists[dists > 0][:k] + 1e-6)
        w = w / w.sum()
        for f in FORMATIONS:
            if f not in sp_df.columns:
                continue
            true_val  = sp_df[f].iloc[i]
            pred_val  = float(np.dot(w, sp_df[f].iloc[idx_use].values))
            errs[f].append((pred_val - true_val)**2)
    print(f"\nk={k}:")
    for f in FORMATIONS:
        rmse_f = float(np.sqrt(np.mean(errs[f])))
        print(f"  {f}: KNN RMSE = {rmse_f:.3f} ft")

# ── 4. TEST WELL XY vs TRAIN WELL XY ─────────────────────────────────────────
print("\n" + "="*60)
print("TEST WELL LOCATION vs TRAIN WELLS")
print("="*60)

for wid in test_ids:
    hw_t = pd.read_csv(TEST_DIR / f"{wid}__horizontal_well.csv")
    tx   = float(hw_t["X"].median())
    ty   = float(hw_t["Y"].median())
    # find nearest train wells
    dists_raw = np.sqrt((sp_df.x_med - tx)**2 + (sp_df.y_med - ty)**2)
    nearest5  = dists_raw.nsmallest(5)
    print(f"\nTest well {wid}:")
    print(f"  XY = ({tx:.0f}, {ty:.0f})")
    print(f"  Nearest train wells (distance in ft):")
    for idx_n, dist_n in nearest5.items():
        print(f"    {sp_df.iloc[idx_n].wid}  dist={dist_n:.0f} ft")

PHYSICS VERIFICATION: TVT = -Z + formation_top + b_well
Applied to EVAL zone using b_well from KNOWN zone
Per-formation RMSE on EVAL zone (using known-zone b_well):
  ANCC: mean=nan  std=nan  max=nan  p95=nan
  ASTNU: mean=0.0070  std=0.0004  max=0.0090  p95=0.0077
  ASTNL: mean=0.0070  std=0.0004  max=0.0092  p95=0.0077
  EGFDU: mean=0.0070  std=0.0004  max=0.0092  p95=0.0077
  EGFDL: mean=nan  std=nan  max=nan  p95=nan
  BUDA: mean=0.0070  std=0.0004  max=0.0091  p95=0.0077

Best formation counts:
best_form
EGFDL    161
EGFDU    159
ANCC     136
ASTNL    120
BUDA     111
ASTNU     86

Best formation RMSE stats:
count    766.0000
mean       0.0069
std        0.0004
min        0.0059
25%        0.0067
50%        0.0069
75%        0.0072
max        0.0090

Wells with best_rmse > 1.0  : 0
Wells with best_rmse > 5.0  : 0
Wells with best_rmse > 10.0 : 0

SPATIAL STRUCTURE OF FORMATION TOPS
(median XY per well + median formation depth)
Spatial X range: 2857774 → 3036387  span=178613 ft
Spat

In [4]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

BASE_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR  = BASE_DIR / "test"

train_hw  = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_ids = [f.stem.replace("__horizontal_well","") for f in train_hw]

FORMATIONS = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']

# ── 1. WITHIN-WELL XY EXTENT ──────────────────────────────────────────────────
print("="*60)
print("WITHIN-WELL XY EXTENT")
print("(how far does each well span laterally?)")
print("="*60)

xy_extents = []
for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    x_span = float(hw["X"].max() - hw["X"].min())
    y_span = float(hw["Y"].max() - hw["Y"].min())
    xy_span = float(np.sqrt(x_span**2 + y_span**2))
    # eval zone extent
    ev = hw[hw["TVT_input"].isna()]
    x_span_ev = float(ev["X"].max() - ev["X"].min())
    y_span_ev = float(ev["Y"].max() - ev["Y"].min())
    xy_span_ev = float(np.sqrt(x_span_ev**2 + y_span_ev**2))
    xy_extents.append({"xy_span_total": xy_span,
                       "xy_span_eval": xy_span_ev,
                       "x_span": x_span, "y_span": y_span})

ex_df = pd.DataFrame(xy_extents)
print(f"Total well XY span  mean±std : {ex_df.xy_span_total.mean():.0f} ± {ex_df.xy_span_total.std():.0f} ft")
print(f"Total well XY span  min/max  : {ex_df.xy_span_total.min():.0f} / {ex_df.xy_span_total.max():.0f} ft")
print(f"Eval zone XY span   mean±std : {ex_df.xy_span_eval.mean():.0f} ± {ex_df.xy_span_eval.std():.0f} ft")
print(f"Eval zone XY span   min/max  : {ex_df.xy_span_eval.min():.0f} / {ex_df.xy_span_eval.max():.0f} ft")

# ── 2. FORMATION TOP VARIATION WITHIN A SINGLE WELL ───────────────────────────
print("\n" + "="*60)
print("FORMATION TOP VARIATION WITHIN ONE WELL")
print("(how much does formation depth change along trajectory?)")
print("="*60)

within_well_var = {f: [] for f in FORMATIONS}
for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    for f in FORMATIONS:
        if f in hw.columns and hw[f].notna().sum() > 10:
            within_well_var[f].append(hw[f].std())

for f in FORMATIONS:
    vals = within_well_var[f]
    if vals:
        print(f"  {f} within-well std: mean={np.mean(vals):.2f}  "
              f"max={np.max(vals):.2f}  "
              f"p95={np.percentile(vals,95):.2f}")

# ── 3. POINT-BY-POINT KNN IMPUTATION ─────────────────────────────────────────
# Use ALL XY points from train wells (sampled) to build a dense spatial grid
# Then predict formation tops at eval zone XY locations
print("\n" + "="*60)
print("POINT-BY-POINT KNN — LOO FORMATION TOP IMPUTATION")
print("(sample 50 pts/well for speed, exclude self-well)")
print("="*60)

# Build dense point cloud from all train wells
SAMPLE_PER_WELL = 50
all_pts = []
for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    idx = np.linspace(0, len(hw)-1, min(SAMPLE_PER_WELL, len(hw)), dtype=int)
    sample = hw.iloc[idx]
    for _, row in sample.iterrows():
        pt = {"wid": wid, "x": row["X"], "y": row["Y"]}
        for f in FORMATIONS:
            pt[f] = row[f] if f in hw.columns else np.nan
        all_pts.append(pt)

pts_df = pd.DataFrame(all_pts)
xy_pts  = pts_df[["x","y"]].values
scale   = xy_pts.std(0); scale[scale<1] = 1
xy_norm = xy_pts / scale
tree    = cKDTree(xy_norm)

# LOO test: for each train well eval zone (first 50 wells for speed)
print("Testing on first 50 train wells (eval zone)...")
pp_rmse = {f: [] for f in FORMATIONS}

for wid in train_ids[:50]:
    hw      = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    ev      = hw[hw["TVT_input"].isna()]
    if len(ev) == 0:
        continue
    # sample 20 eval zone points
    idx_ev  = np.linspace(0, len(ev)-1, min(20, len(ev)), dtype=int)
    ev_s    = ev.iloc[idx_ev]
    xy_q    = ev_s[["X","Y"]].values / scale
    dists, idxs = tree.query(xy_q, k=15)

    for fi, f in enumerate(FORMATIONS):
        if f not in hw.columns:
            continue
        preds = []
        trues = ev_s[f].values
        for qi in range(len(ev_s)):
            nb_idx  = idxs[qi]
            nb_dist = dists[qi]
            # exclude self-well points
            mask    = pts_df.iloc[nb_idx]["wid"].values != wid
            nb_idx2 = nb_idx[mask][:5]
            nb_dist2= nb_dist[mask][:5]
            if len(nb_idx2) == 0:
                preds.append(np.nan)
                continue
            w = 1.0 / (nb_dist2 + 1e-6)
            w = w / w.sum()
            pred = float(np.dot(w, pts_df.iloc[nb_idx2][f].values))
            preds.append(pred)
        preds = np.array(preds)
        trues = np.array(trues, dtype=float)
        valid = np.isfinite(preds) & np.isfinite(trues)
        if valid.sum() > 3:
            rmse = float(np.sqrt(np.mean((preds[valid]-trues[valid])**2)))
            pp_rmse[f].append(rmse)

print("Point-by-point KNN (k=5, excl. self-well) RMSE:")
for f in FORMATIONS:
    if pp_rmse[f]:
        print(f"  {f}: mean={np.mean(pp_rmse[f]):.3f}  "
              f"std={np.std(pp_rmse[f]):.3f}  "
              f"max={np.max(pp_rmse[f]):.3f}")

# ── 4. LOCAL PLANE FIT IMPUTATION ─────────────────────────────────────────────
print("\n" + "="*60)
print("LOCAL PLANE FIT — formation top imputation")
print("(fits ax+by+c plane through k nearest neighbors)")
print("="*60)

plane_rmse = {f: [] for f in FORMATIONS}
for wid in train_ids[:50]:
    hw   = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    ev   = hw[hw["TVT_input"].isna()]
    if len(ev) == 0:
        continue
    idx_ev = np.linspace(0, len(ev)-1, min(20, len(ev)), dtype=int)
    ev_s   = ev.iloc[idx_ev]
    xy_q   = ev_s[["X","Y"]].values / scale
    dists, idxs = tree.query(xy_q, k=30)

    for f in FORMATIONS:
        if f not in hw.columns:
            continue
        preds = []
        trues = ev_s[f].values
        for qi in range(len(ev_s)):
            nb_idx  = idxs[qi]
            nb_dist = dists[qi]
            mask    = pts_df.iloc[nb_idx]["wid"].values != wid
            nb_idx2 = nb_idx[mask][:20]
            if len(nb_idx2) < 4:
                preds.append(np.nan); continue
            nb_xy   = xy_pts[nb_idx2]
            nb_z    = pts_df.iloc[nb_idx2][f].values.astype(float)
            valid_m = np.isfinite(nb_z)
            if valid_m.sum() < 4:
                preds.append(np.nan); continue
            # fit plane: z = ax + by + c
            A = np.column_stack([nb_xy[valid_m,0],
                                 nb_xy[valid_m,1],
                                 np.ones(valid_m.sum())])
            try:
                coef, _, _, _ = np.linalg.lstsq(A, nb_z[valid_m], rcond=None)
                qx, qy = ev_s.iloc[qi]["X"], ev_s.iloc[qi]["Y"]
                pred   = coef[0]*qx + coef[1]*qy + coef[2]
                preds.append(pred)
            except:
                preds.append(np.nan)

        preds = np.array(preds)
        trues = np.array(trues, dtype=float)
        valid = np.isfinite(preds) & np.isfinite(trues)
        if valid.sum() > 3:
            rmse = float(np.sqrt(np.mean((preds[valid]-trues[valid])**2)))
            plane_rmse[f].append(rmse)

print("Local plane fit (k=20, excl. self-well) RMSE:")
for f in FORMATIONS:
    if plane_rmse[f]:
        print(f"  {f}: mean={np.mean(plane_rmse[f]):.3f}  "
              f"std={np.std(plane_rmse[f]):.3f}  "
              f"max={np.max(plane_rmse[f]):.3f}")

WITHIN-WELL XY EXTENT
(how far does each well span laterally?)
Total well XY span  mean±std : 6124 ± 1295 ft
Total well XY span  min/max  : 1529 / 11566 ft
Eval zone XY span   mean±std : 4886 ± 1299 ft
Eval zone XY span   min/max  : 405 / 10040 ft

FORMATION TOP VARIATION WITHIN ONE WELL
(how much does formation depth change along trajectory?)
  ANCC within-well std: mean=63.67  max=157.25  p95=101.84
  ASTNU within-well std: mean=63.37  max=157.25  p95=101.83
  ASTNL within-well std: mean=63.37  max=157.25  p95=101.83
  EGFDU within-well std: mean=63.37  max=157.25  p95=101.83
  EGFDL within-well std: mean=63.38  max=157.25  p95=101.83
  BUDA within-well std: mean=63.37  max=157.25  p95=101.83

POINT-BY-POINT KNN — LOO FORMATION TOP IMPUTATION
(sample 50 pts/well for speed, exclude self-well)
Testing on first 50 train wells (eval zone)...
Point-by-point KNN (k=5, excl. self-well) RMSE:
  ANCC: mean=15.738  std=12.037  max=63.197
  ASTNU: mean=16.777  std=13.552  max=64.410
  ASTNL: me

In [5]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

BASE_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
TRAIN_DIR = BASE_DIR / "train"

train_hw  = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_ids = [f.stem.replace("__horizontal_well","") for f in train_hw]

FORMATIONS = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']

# ── 1. COMPUTE PER-WELL DIP (formation top gradient dF/dX, dF/dY) ────────────
print("="*60)
print("STEP 1: COMPUTE LOCAL DIP PER WELL")
print("dip = d(formation_top)/d(lateral_distance)")
print("="*60)

dip_records = []
for wid in train_ids:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    # Use ENTIRE well (known + eval — both have formation tops in train)
    x  = hw["X"].values
    y  = hw["Y"].values
    rec = {"wid": wid,
           "x_start": float(x[0]),  "y_start": float(y[0]),
           "x_end"  : float(x[-1]), "y_end"  : float(y[-1]),
           "x_med"  : float(np.median(x)),
           "y_med"  : float(np.median(y))}
    for f in FORMATIONS:
        if f not in hw.columns:
            rec[f"dip_x_{f}"] = np.nan
            rec[f"dip_y_{f}"] = np.nan
            rec[f"f_start_{f}"] = np.nan
            continue
        fval = hw[f].values
        valid = np.isfinite(fval) & np.isfinite(x) & np.isfinite(y)
        if valid.sum() < 10:
            rec[f"dip_x_{f}"] = np.nan
            rec[f"dip_y_{f}"] = np.nan
            rec[f"f_start_{f}"] = np.nan
            continue
        # Fit: f = dip_x * X + dip_y * Y + c
        A = np.column_stack([x[valid], y[valid], np.ones(valid.sum())])
        coef, _, _, _ = np.linalg.lstsq(A, fval[valid], rcond=None)
        rec[f"dip_x_{f}"] = float(coef[0])
        rec[f"dip_y_{f}"] = float(coef[1])
        rec[f"f_start_{f}"] = float(fval[0]) if np.isfinite(fval[0]) else np.nan
    dip_records.append(rec)

dip_df = pd.DataFrame(dip_records)

print("Dip statistics across all wells:")
for f in FORMATIONS:
    col_x = f"dip_x_{f}"
    col_y = f"dip_y_{f}"
    if col_x in dip_df.columns:
        dx = dip_df[col_x].dropna()
        dy = dip_df[col_y].dropna()
        print(f"  {f}: dip_x mean={dx.mean():.5f} std={dx.std():.5f} | "
              f"dip_y mean={dy.mean():.5f} std={dy.std():.5f}")

# ── 2. DIP-ANCHORED EXTRAPOLATION — LOO TEST ─────────────────────────────────
print("\n" + "="*60)
print("STEP 2: DIP-ANCHORED EXTRAPOLATION — LOO TEST")
print("TVT = -Z + (f_anchor + dip_x*ΔX + dip_y*ΔY) + b_well")
print("="*60)

# Build KD-tree on well median XY for dip lookup
xy_med  = dip_df[["x_med","y_med"]].values
scale   = xy_med.std(0); scale[scale<1] = 1
xy_norm = xy_med / scale
tree    = cKDTree(xy_norm)

rmse_dip   = {f: [] for f in FORMATIONS}
rmse_knn   = {f: [] for f in FORMATIONS}  # baseline comparison

for wid in train_ids[:100]:  # test on 100 wells
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_known = hw["TVT_input"].notna()
    mask_eval  = hw["TVT_input"].isna()
    kn = hw[mask_known]
    ev = hw[mask_eval]
    if len(ev) == 0 or len(kn) < 5:
        continue

    # Anchor: last known zone point
    anchor_x = float(kn.iloc[-1]["X"])
    anchor_y = float(kn.iloc[-1]["Y"])

    # Find this well's index in dip_df
    self_idx = dip_df.index[dip_df["wid"] == wid]
    if len(self_idx) == 0:
        continue
    self_i = self_idx[0]

    # KNN neighbors (exclude self)
    q = np.array([[dip_df.iloc[self_i]["x_med"],
                   dip_df.iloc[self_i]["y_med"]]]) / scale
    dists, idxs = tree.query(q, k=11)
    nb_idx = idxs[0][idxs[0] != self_i][:10]

    # Eval zone XY and true formation tops
    ev_x = ev["X"].values
    ev_y = ev["Y"].values
    dX   = ev_x - anchor_x
    dY   = ev_y - anchor_y

    for f in FORMATIONS:
        if f not in hw.columns:
            continue

        # -- DIP-ANCHORED approach --
        # Anchor formation top at last known point
        anchor_f = float(kn.iloc[-1][f]) if np.isfinite(kn.iloc[-1][f]) else np.nan
        if np.isnan(anchor_f):
            continue

        # Get neighbor dips (exclude self)
        nb_dip_x = dip_df.iloc[nb_idx][f"dip_x_{f}"].dropna().values
        nb_dip_y = dip_df.iloc[nb_idx][f"dip_y_{f}"].dropna().values
        if len(nb_dip_x) == 0:
            continue

        # Weighted average dip from neighbors
        nb_dists  = dists[0][idxs[0] != self_i][:len(nb_dip_x)]
        w = 1.0 / (nb_dists[:len(nb_dip_x)] + 1e-6)
        w = w / w.sum()
        pred_dip_x = float(np.dot(w[:len(nb_dip_x)], nb_dip_x))
        pred_dip_y = float(np.dot(w[:len(nb_dip_y)], nb_dip_y))

        # Extrapolate formation top
        f_pred_dip = anchor_f + pred_dip_x * dX + pred_dip_y * dY

        # True formation tops in eval zone
        f_true = ev[f].values

        # Compute RMSE
        valid = np.isfinite(f_pred_dip) & np.isfinite(f_true)
        if valid.sum() > 5:
            rmse_d = float(np.sqrt(np.mean((f_pred_dip[valid] - f_true[valid])**2)))
            rmse_dip[f].append(rmse_d)

        # -- KNN ABSOLUTE baseline (for comparison) --
        nb_f_vals = dip_df.iloc[nb_idx][f"f_start_{f}"].dropna().values
        if len(nb_f_vals) > 0:
            f_pred_knn = float(np.mean(nb_f_vals))
            rmse_k = float(np.sqrt(np.mean((f_pred_knn - f_true)**2)))
            rmse_knn[f].append(rmse_k)

print("Formation top imputation RMSE comparison (100 wells):")
print(f"{'Formation':<8} {'Dip-Anchored':>14} {'KNN-Absolute':>14} {'Improvement':>12}")
print("-" * 52)
for f in FORMATIONS:
    d = np.mean(rmse_dip[f]) if rmse_dip[f] else np.nan
    k = np.mean(rmse_knn[f]) if rmse_knn[f] else np.nan
    imp = (k-d)/k*100 if (not np.isnan(d) and not np.isnan(k) and k>0) else np.nan
    print(f"{f:<8} {d:>14.3f} {k:>14.3f} {imp:>11.1f}%")

# ── 3. DIP-ANCHORED TVT RMSE ─────────────────────────────────────────────────
print("\n" + "="*60)
print("STEP 3: DIP-ANCHORED TVT PREDICTION RMSE")
print("TVT = -Z + f_pred_dip + b_well")
print("="*60)

tvt_rmse_dip = []
tvt_rmse_knn = []

for wid in train_ids[:100]:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_known = hw["TVT_input"].notna()
    mask_eval  = hw["TVT_input"].isna()
    kn = hw[mask_known]
    ev = hw[mask_eval]
    if len(ev) == 0 or len(kn) < 5:
        continue

    self_idx = dip_df.index[dip_df["wid"] == wid]
    if len(self_idx) == 0:
        continue
    self_i = self_idx[0]

    q = np.array([[dip_df.iloc[self_i]["x_med"],
                   dip_df.iloc[self_i]["y_med"]]]) / scale
    dists, idxs = tree.query(q, k=11)
    nb_idx = idxs[0][idxs[0] != self_i][:10]

    anchor_x = float(kn.iloc[-1]["X"])
    anchor_y = float(kn.iloc[-1]["Y"])
    dX = ev["X"].values - anchor_x
    dY = ev["Y"].values - anchor_y

    # Use best formation (EGFDU as default)
    best_rmse = np.inf
    best_pred = None

    for f in FORMATIONS:
        if f not in hw.columns:
            continue
        anchor_f = float(kn.iloc[-1][f])
        if np.isnan(anchor_f):
            continue
        b_well = float(np.mean(kn["TVT"].values + kn["Z"].values - kn[f].values))

        nb_dip_x = dip_df.iloc[nb_idx][f"dip_x_{f}"].dropna().values
        nb_dip_y = dip_df.iloc[nb_idx][f"dip_y_{f}"].dropna().values
        if len(nb_dip_x) == 0:
            continue
        nb_dists = dists[0][idxs[0] != self_i][:len(nb_dip_x)]
        w = 1.0 / (nb_dists[:len(nb_dip_x)] + 1e-6)
        w = w / w.sum()
        pred_dip_x = float(np.dot(w[:len(nb_dip_x)], nb_dip_x))
        pred_dip_y = float(np.dot(w[:len(nb_dip_y)], nb_dip_y))

        f_pred = anchor_f + pred_dip_x * dX + pred_dip_y * dY
        tvt_pred = -ev["Z"].values + f_pred + b_well
        tvt_true = ev["TVT"].values
        valid = np.isfinite(tvt_pred) & np.isfinite(tvt_true)
        if valid.sum() > 5:
            rmse = float(np.sqrt(np.mean((tvt_pred[valid]-tvt_true[valid])**2)))
            if rmse < best_rmse:
                best_rmse = rmse
                best_pred = tvt_pred

    if best_pred is not None:
        tvt_rmse_dip.append(best_rmse)

print(f"Dip-anchored TVT RMSE (best formation, 100 wells):")
print(f"  mean : {np.mean(tvt_rmse_dip):.4f} ft")
print(f"  std  : {np.std(tvt_rmse_dip):.4f} ft")
print(f"  p50  : {np.median(tvt_rmse_dip):.4f} ft")
print(f"  p95  : {np.percentile(tvt_rmse_dip,95):.4f} ft")
print(f"  max  : {np.max(tvt_rmse_dip):.4f} ft")
print(f"\n  Wells with RMSE > 5   : {sum(r>5 for r in tvt_rmse_dip)}")
print(f"  Wells with RMSE > 10  : {sum(r>10 for r in tvt_rmse_dip)}")
print(f"  Wells with RMSE > 20  : {sum(r>20 for r in tvt_rmse_dip)}")
print(f"\nNOTE: This simulates what happens on REAL test wells")
print(f"where formation tops are NOT given.")

STEP 1: COMPUTE LOCAL DIP PER WELL
dip = d(formation_top)/d(lateral_distance)
Dip statistics across all wells:
  ANCC: dip_x mean=-0.02105 std=0.13078 | dip_y mean=0.02648 std=0.10363
  ASTNU: dip_x mean=-0.02148 std=0.13089 | dip_y mean=0.02586 std=0.10451
  ASTNL: dip_x mean=-0.02149 std=0.13089 | dip_y mean=0.02585 std=0.10451
  EGFDU: dip_x mean=-0.02151 std=0.13089 | dip_y mean=0.02584 std=0.10451
  EGFDL: dip_x mean=-0.02140 std=0.13094 | dip_y mean=0.02588 std=0.10458
  BUDA: dip_x mean=-0.02153 std=0.13088 | dip_y mean=0.02583 std=0.10451

STEP 2: DIP-ANCHORED EXTRAPOLATION — LOO TEST
TVT = -Z + (f_anchor + dip_x*ΔX + dip_y*ΔY) + b_well
Formation top imputation RMSE comparison (100 wells):
Formation   Dip-Anchored   KNN-Absolute  Improvement
----------------------------------------------------
ANCC             14.006        118.956        88.2%
ASTNU            14.500        118.299        87.7%
ASTNL            14.500        116.848        87.6%
EGFDU            14.500        

In [6]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

BASE_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
TRAIN_DIR = BASE_DIR / "train"

train_hw  = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_ids = [f.stem.replace("__horizontal_well","") for f in train_hw]

FORMATIONS = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']

# ── REBUILD DIP TABLE (needed for neighbor lookup) ────────────────────────────
dip_records = []
for wid in train_ids:
    hw  = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    x   = hw["X"].values; y = hw["Y"].values
    rec = {"wid": wid,
           "x_med": float(np.median(x)),
           "y_med": float(np.median(y))}
    for f in FORMATIONS:
        if f not in hw.columns:
            rec[f"dip_x_{f}"] = np.nan
            rec[f"dip_y_{f}"] = np.nan
            continue
        fval  = hw[f].values
        valid = np.isfinite(fval) & np.isfinite(x) & np.isfinite(y)
        if valid.sum() < 10:
            rec[f"dip_x_{f}"] = np.nan
            rec[f"dip_y_{f}"] = np.nan
            continue
        A    = np.column_stack([x[valid], y[valid], np.ones(valid.sum())])
        coef, _, _, _ = np.linalg.lstsq(A, fval[valid], rcond=None)
        rec[f"dip_x_{f}"] = float(coef[0])
        rec[f"dip_y_{f}"] = float(coef[1])
    dip_records.append(rec)

dip_df  = pd.DataFrame(dip_records)
xy_med  = dip_df[["x_med","y_med"]].values
scale   = xy_med.std(0); scale[scale<1] = 1
xy_norm = xy_med / scale
tree    = cKDTree(xy_norm)

# ── 1. OWN-WELL LOCAL DIP FROM TAIL OF KNOWN ZONE ────────────────────────────
print("="*60)
print("APPROACH A: OWN-WELL LOCAL DIP")
print("Use formation top gradient from last N rows of known zone")
print("="*60)

def compute_local_dip(x, y, fval, tail_rows):
    """Fit plane to last tail_rows of known zone."""
    x_t = x[-tail_rows:]; y_t = y[-tail_rows:]; f_t = fval[-tail_rows:]
    valid = np.isfinite(f_t) & np.isfinite(x_t) & np.isfinite(y_t)
    if valid.sum() < 4:
        return np.nan, np.nan
    A    = np.column_stack([x_t[valid], y_t[valid], np.ones(valid.sum())])
    coef, _, _, _ = np.linalg.lstsq(A, f_t[valid], rcond=None)
    return float(coef[0]), float(coef[1])

results = []
for tail in [50, 100, 200, 500]:
    rmse_list = []
    for wid in train_ids[:200]:
        hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
        mask_known = hw["TVT_input"].notna()
        mask_eval  = hw["TVT_input"].isna()
        kn = hw[mask_known]; ev = hw[mask_eval]
        if len(ev) == 0 or len(kn) < max(tail, 10):
            continue

        kn_x = kn["X"].values; kn_y = kn["Y"].values
        ev_x = ev["X"].values; ev_y = ev["Y"].values
        anchor_x = float(kn.iloc[-1]["X"])
        anchor_y = float(kn.iloc[-1]["Y"])
        dX = ev_x - anchor_x
        dY = ev_y - anchor_y

        best_rmse = np.inf
        for f in FORMATIONS:
            if f not in hw.columns: continue
            kn_f  = kn[f].values
            anchor_f = float(kn.iloc[-1][f])
            if np.isnan(anchor_f): continue
            b_well = float(np.nanmean(kn["TVT"].values + kn["Z"].values - kn_f))

            dx_l, dy_l = compute_local_dip(kn_x, kn_y, kn_f, tail)
            if np.isnan(dx_l): continue

            f_pred   = anchor_f + dx_l * dX + dy_l * dY
            tvt_pred = -ev["Z"].values + f_pred + b_well
            tvt_true = ev["TVT"].values
            valid    = np.isfinite(tvt_pred) & np.isfinite(tvt_true)
            if valid.sum() > 5:
                rmse = float(np.sqrt(np.mean((tvt_pred[valid]-tvt_true[valid])**2)))
                if rmse < best_rmse:
                    best_rmse = rmse
        if best_rmse < np.inf:
            rmse_list.append(best_rmse)

    print(f"tail={tail:>4} rows | mean={np.mean(rmse_list):.3f}  "
          f"std={np.std(rmse_list):.3f}  "
          f"p50={np.median(rmse_list):.3f}  "
          f"p95={np.percentile(rmse_list,95):.3f}  "
          f">5ft={sum(r>5 for r in rmse_list)}/200")

# ── 2. HYBRID: OWN DIP + NEIGHBOR DIP BLEND ──────────────────────────────────
print("\n" + "="*60)
print("APPROACH B: HYBRID — own-well dip + neighbor dip blend")
print("alpha * own_dip + (1-alpha) * neighbor_dip")
print("="*60)

for alpha in [0.0, 0.3, 0.5, 0.7, 0.9, 1.0]:
    rmse_list = []
    for wid in train_ids[:200]:
        hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
        mask_known = hw["TVT_input"].notna()
        mask_eval  = hw["TVT_input"].isna()
        kn = hw[mask_known]; ev = hw[mask_eval]
        if len(ev) == 0 or len(kn) < 100: continue

        kn_x = kn["X"].values; kn_y = kn["Y"].values
        ev_x = ev["X"].values; ev_y = ev["Y"].values
        anchor_x = float(kn.iloc[-1]["X"])
        anchor_y = float(kn.iloc[-1]["Y"])
        dX = ev_x - anchor_x; dY = ev_y - anchor_y

        self_idx = dip_df.index[dip_df["wid"] == wid]
        if len(self_idx) == 0: continue
        self_i = self_idx[0]
        q = np.array([[dip_df.iloc[self_i]["x_med"],
                       dip_df.iloc[self_i]["y_med"]]]) / scale
        dists, idxs = tree.query(q, k=11)
        nb_idx = idxs[0][idxs[0] != self_i][:10]
        nb_dists = dists[0][idxs[0] != self_i][:10]

        best_rmse = np.inf
        for f in FORMATIONS:
            if f not in hw.columns: continue
            kn_f     = kn[f].values
            anchor_f = float(kn.iloc[-1][f])
            if np.isnan(anchor_f): continue
            b_well   = float(np.nanmean(kn["TVT"].values + kn["Z"].values - kn_f))

            # Own dip (tail=200)
            dx_own, dy_own = compute_local_dip(kn_x, kn_y, kn_f, 200)
            if np.isnan(dx_own): continue

            # Neighbor dip
            nb_dx = dip_df.iloc[nb_idx][f"dip_x_{f}"].dropna().values
            nb_dy = dip_df.iloc[nb_idx][f"dip_y_{f}"].dropna().values
            if len(nb_dx) == 0: continue
            w = 1.0/(nb_dists[:len(nb_dx)]+1e-6); w = w/w.sum()
            dx_nb = float(np.dot(w[:len(nb_dx)], nb_dx))
            dy_nb = float(np.dot(w[:len(nb_dy)], nb_dy))

            # Blend
            dx_blend = alpha * dx_own + (1-alpha) * dx_nb
            dy_blend = alpha * dy_own + (1-alpha) * dy_nb

            f_pred   = anchor_f + dx_blend * dX + dy_blend * dY
            tvt_pred = -ev["Z"].values + f_pred + b_well
            tvt_true = ev["TVT"].values
            valid    = np.isfinite(tvt_pred) & np.isfinite(tvt_true)
            if valid.sum() > 5:
                rmse = float(np.sqrt(np.mean((tvt_pred[valid]-tvt_true[valid])**2)))
                if rmse < best_rmse:
                    best_rmse = rmse
        if best_rmse < np.inf:
            rmse_list.append(best_rmse)

    print(f"alpha={alpha:.1f} (own) | mean={np.mean(rmse_list):.3f}  "
          f"p50={np.median(rmse_list):.3f}  "
          f"p95={np.percentile(rmse_list,95):.3f}  "
          f">5ft={sum(r>5 for r in rmse_list)}/200")

# ── 3. DIAGNOSE HARD WELLS ────────────────────────────────────────────────────
print("\n" + "="*60)
print("APPROACH C: WHAT MAKES HARD WELLS HARD?")
print("(using best alpha from above, tail=200)")
print("="*60)

hard_well_stats = []
for wid in train_ids[:200]:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_known = hw["TVT_input"].notna()
    mask_eval  = hw["TVT_input"].isna()
    kn = hw[mask_known]; ev = hw[mask_eval]
    if len(ev) == 0 or len(kn) < 100: continue

    kn_x = kn["X"].values; kn_y = kn["Y"].values
    anchor_x = float(kn.iloc[-1]["X"]); anchor_y = float(kn.iloc[-1]["Y"])
    dX = ev["X"].values - anchor_x; dY = ev["Y"].values - anchor_y

    self_idx = dip_df.index[dip_df["wid"] == wid]
    if len(self_idx) == 0: continue
    self_i = self_idx[0]
    q = np.array([[dip_df.iloc[self_i]["x_med"],
                   dip_df.iloc[self_i]["y_med"]]]) / scale
    dists, idxs = tree.query(q, k=11)
    nb_idx  = idxs[0][idxs[0] != self_i][:10]
    nb_dist = dists[0][idxs[0] != self_i][:10]

    f = "EGFDU"
    if f not in hw.columns: continue
    kn_f     = kn[f].values
    anchor_f = float(kn.iloc[-1][f])
    if np.isnan(anchor_f): continue
    b_well   = float(np.nanmean(kn["TVT"].values + kn["Z"].values - kn_f))

    dx_own, dy_own = compute_local_dip(kn_x, kn_y, kn_f, 200)
    if np.isnan(dx_own): continue

    nb_dx = dip_df.iloc[nb_idx][f"dip_x_{f}"].dropna().values
    nb_dy = dip_df.iloc[nb_idx][f"dip_y_{f}"].dropna().values
    if len(nb_dx) == 0: continue
    w = 1.0/(nb_dist[:len(nb_dx)]+1e-6); w = w/w.sum()
    dx_nb = float(np.dot(w[:len(nb_dx)], nb_dx))
    dy_nb = float(np.dot(w[:len(nb_dy)], nb_dy))

    # alpha=0.7
    dx_b = 0.7*dx_own + 0.3*dx_nb
    dy_b = 0.7*dy_own + 0.3*dy_nb

    f_pred   = anchor_f + dx_b * dX + dy_b * dY
    tvt_pred = -ev["Z"].values + f_pred + b_well
    tvt_true = ev["TVT"].values
    valid    = np.isfinite(tvt_pred) & np.isfinite(tvt_true)
    if valid.sum() > 5:
        rmse = float(np.sqrt(np.mean((tvt_pred[valid]-tvt_true[valid])**2)))
        eval_span = float(np.sqrt((ev["X"].values[-1]-ev["X"].values[0])**2 +
                                   (ev["Y"].values[-1]-ev["Y"].values[0])**2))
        gr_miss   = float(ev["GR"].isna().mean())
        dip_mag   = float(np.sqrt(dx_own**2 + dy_own**2))
        hard_well_stats.append({
            "wid": wid, "rmse": rmse,
            "eval_span": eval_span,
            "gr_miss_eval": gr_miss,
            "dip_mag": dip_mag,
            "n_known": len(kn),
            "n_eval": len(ev),
            "nearest_dist": float(nb_dist[0])
        })

hw_df = pd.DataFrame(hard_well_stats)
print(f"Total wells analyzed: {len(hw_df)}")
print(f"\nCorrelation with RMSE:")
for col in ["eval_span","gr_miss_eval","dip_mag","n_known","n_eval","nearest_dist"]:
    corr = float(hw_df["rmse"].corr(hw_df[col]))
    print(f"  {col:<20}: r = {corr:+.3f}")

print(f"\nTop 10 hardest wells:")
print(hw_df.nlargest(10,"rmse")[
    ["wid","rmse","eval_span","gr_miss_eval","dip_mag","nearest_dist"]
].round(2).to_string(index=False))

APPROACH A: OWN-WELL LOCAL DIP
Use formation top gradient from last N rows of known zone
tail=  50 rows | mean=28.043  std=25.452  p50=22.291  p95=73.469  >5ft=176/200
tail= 100 rows | mean=31.498  std=49.522  p50=22.680  p95=80.884  >5ft=180/200
tail= 200 rows | mean=30.881  std=52.046  p50=20.837  p95=75.215  >5ft=186/200
tail= 500 rows | mean=30.798  std=36.652  p50=24.117  p95=65.410  >5ft=192/200

APPROACH B: HYBRID — own-well dip + neighbor dip blend
alpha * own_dip + (1-alpha) * neighbor_dip
alpha=0.0 (own) | mean=14.647  p50=11.064  p95=35.835  >5ft=170/200
alpha=0.3 (own) | mean=16.030  p50=11.910  p95=36.409  >5ft=176/200
alpha=0.5 (own) | mean=19.293  p50=14.726  p95=43.978  >5ft=181/200
alpha=0.7 (own) | mean=23.396  p50=17.975  p95=56.321  >5ft=182/200
alpha=0.9 (own) | mean=28.284  p50=19.758  p95=68.990  >5ft=186/200
alpha=1.0 (own) | mean=30.881  p50=20.837  p95=75.215  >5ft=186/200

APPROACH C: WHAT MAKES HARD WELLS HARD?
(using best alpha from above, tail=200)
Total w

In [7]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree

BASE_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
TRAIN_DIR = BASE_DIR / "train"

train_hw  = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_ids = [f.stem.replace("__horizontal_well","") for f in train_hw]
FORMATIONS = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']

# ── BUILD DENSE SPATIAL POINT CLOUD ──────────────────────────────────────────
print("Building dense spatial point cloud...")
SAMPLE_PER_WELL = 100
all_pts = []
for wid in train_ids:
    hw  = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    idx = np.linspace(0, len(hw)-1, min(SAMPLE_PER_WELL, len(hw)), dtype=int)
    s   = hw.iloc[idx]
    for i in range(len(s)):
        row = s.iloc[i]
        pt  = {"wid": wid, "x": float(row["X"]), "y": float(row["Y"])}
        for f in FORMATIONS:
            pt[f] = float(row[f]) if f in hw.columns and np.isfinite(row[f]) else np.nan
        all_pts.append(pt)

pts_df  = pd.DataFrame(all_pts)
xy_pts  = pts_df[["x","y"]].values.astype(np.float64)
scale   = xy_pts.std(0); scale[scale < 1] = 1.0
xy_norm = xy_pts / scale
tree    = cKDTree(xy_norm)
print(f"Dense cloud: {len(pts_df)} points from {len(train_ids)} wells")

# ── ANCHOR-CORRECTED DENSE KNN ────────────────────────────────────────────────
def predict_formation_anchored(ev_x, ev_y, anchor_x, anchor_y,
                                anchor_f_true, wid, f, k=10):
    """
    For each eval point (ex, ey):
      1. KNN from dense cloud (exclude self-well)
      2. Predict f_global(x,y)
      3. Correct: f_pred = f_global(x,y) + [anchor_true - f_global(anchor)]
    """
    # Predict at anchor using KNN
    q_anc  = np.array([[anchor_x, anchor_y]]) / scale
    d_a, i_a = tree.query(q_anc, k=k+20)
    mask_a = pts_df["wid"].iloc[i_a[0]].values != wid
    i_a2   = i_a[0][mask_a][:k]
    d_a2   = d_a[0][mask_a][:k]
    if len(i_a2) == 0: return None
    w_a    = 1.0/(d_a2+1e-6); w_a /= w_a.sum()
    f_anc_pred = float(np.dot(w_a, pts_df[f].iloc[i_a2].fillna(np.nan).values))
    if not np.isfinite(f_anc_pred): return None
    anchor_correction = anchor_f_true - f_anc_pred

    # Predict at each eval point
    n_ev = len(ev_x)
    preds = np.full(n_ev, np.nan)
    q_ev  = np.column_stack([ev_x, ev_y]) / scale
    d_ev, i_ev = tree.query(q_ev, k=k+20)
    wids_all = pts_df["wid"].values

    for j in range(n_ev):
        mask_j = wids_all[i_ev[j]] != wid
        ij2    = i_ev[j][mask_j][:k]
        dj2    = d_ev[j][mask_j][:k]
        if len(ij2) == 0: continue
        wj     = 1.0/(dj2+1e-6); wj /= wj.sum()
        fvals  = pts_df[f].iloc[ij2].values.astype(float)
        valid  = np.isfinite(fvals)
        if valid.sum() == 0: continue
        f_pred_j = float(np.dot(wj[valid]/wj[valid].sum(), fvals[valid]))
        preds[j] = f_pred_j + anchor_correction

    return preds

# ── LOO TEST: ANCHOR-CORRECTED vs PLAIN KNN vs DIP-ANCHORED ──────────────────
print("\n" + "="*60)
print("TVT RMSE COMPARISON — 200 wells LOO test")
print("="*60)

results = []
for wid in train_ids[:200]:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    mask_known = hw["TVT_input"].notna()
    mask_eval  = hw["TVT_input"].isna()
    kn = hw[mask_known]; ev = hw[mask_eval]
    if len(ev) == 0 or len(kn) < 10: continue

    anchor_x = float(kn.iloc[-1]["X"])
    anchor_y = float(kn.iloc[-1]["Y"])
    ev_x     = ev["X"].values.astype(np.float64)
    ev_y     = ev["Y"].values.astype(np.float64)
    tvt_true = ev["TVT"].values

    best_rmse_anc  = np.inf
    best_rmse_plain = np.inf

    for f in FORMATIONS:
        if f not in hw.columns: continue
        anchor_f = float(kn.iloc[-1][f])
        if not np.isfinite(anchor_f): continue
        b_well = float(np.nanmean(kn["TVT"].values + kn["Z"].values - kn[f].values))

        # ── Method A: Anchor-corrected KNN ──
        f_pred_anc = predict_formation_anchored(
            ev_x, ev_y, anchor_x, anchor_y, anchor_f, wid, f, k=10)
        if f_pred_anc is not None:
            tvt_pred_anc = -ev["Z"].values + f_pred_anc + b_well
            valid = np.isfinite(tvt_pred_anc) & np.isfinite(tvt_true)
            if valid.sum() > 5:
                rmse = float(np.sqrt(np.mean((tvt_pred_anc[valid]-tvt_true[valid])**2)))
                best_rmse_anc = min(best_rmse_anc, rmse)

        # ── Method B: Plain KNN (no anchor correction) ──
        q_ev  = np.column_stack([ev_x, ev_y]) / scale
        d_ev, i_ev = tree.query(q_ev[:min(10,len(q_ev))], k=15)
        wids_arr = pts_df["wid"].values
        f_pred_plain = []
        for j in range(min(10, len(ev))):
            mask_j = wids_arr[i_ev[j]] != wid
            ij2    = i_ev[j][mask_j][:10]
            dj2    = d_ev[j][mask_j][:10]
            if len(ij2) == 0: f_pred_plain.append(np.nan); continue
            wj = 1.0/(dj2+1e-6); wj /= wj.sum()
            fv = pts_df[f].iloc[ij2].values.astype(float)
            vm = np.isfinite(fv)
            if vm.sum() == 0: f_pred_plain.append(np.nan); continue
            f_pred_plain.append(float(np.dot(wj[vm]/wj[vm].sum(), fv[vm])))
        f_pred_plain = np.array(f_pred_plain)
        tvt_pred_plain = -ev["Z"].values[:len(f_pred_plain)] + f_pred_plain + b_well
        tvt_true_plain = tvt_true[:len(f_pred_plain)]
        valid_p = np.isfinite(tvt_pred_plain) & np.isfinite(tvt_true_plain)
        if valid_p.sum() > 3:
            rmse_p = float(np.sqrt(np.mean((tvt_pred_plain[valid_p]-tvt_true_plain[valid_p])**2)))
            best_rmse_plain = min(best_rmse_plain, rmse_p)

    results.append({
        "wid": wid,
        "rmse_anchored": best_rmse_anc if best_rmse_anc < np.inf else np.nan,
        "rmse_plain":    best_rmse_plain if best_rmse_plain < np.inf else np.nan
    })

res_df = pd.DataFrame(results).dropna()
print(f"Wells tested: {len(res_df)}")
print(f"\nAnchor-corrected KNN:")
print(f"  mean  : {res_df.rmse_anchored.mean():.4f}")
print(f"  std   : {res_df.rmse_anchored.std():.4f}")
print(f"  p50   : {res_df.rmse_anchored.median():.4f}")
print(f"  p95   : {res_df.rmse_anchored.quantile(0.95):.4f}")
print(f"  max   : {res_df.rmse_anchored.max():.4f}")
print(f"  >5ft  : {(res_df.rmse_anchored>5).sum()}")
print(f"  >10ft : {(res_df.rmse_anchored>10).sum()}")

print(f"\nPlain KNN (no anchor):")
print(f"  mean  : {res_df.rmse_plain.mean():.4f}")
print(f"  p50   : {res_df.rmse_plain.median():.4f}")

print(f"\nImprovement: "
      f"{(res_df.rmse_plain.mean()-res_df.rmse_anchored.mean())/res_df.rmse_plain.mean()*100:.1f}%")

# ── GR TYPEWELL CORRELATION (FIXED) ──────────────────────────────────────────
print("\n" + "="*60)
print("GR vs TYPEWELL CORRELATION IN KNOWN ZONE")
print("="*60)

gr_corr_list = []
for wid in train_ids[:150]:
    hw = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    tw = pd.read_csv(TRAIN_DIR / f"{wid}__typewell.csv").sort_values("TVT")
    mask_known = hw["TVT_input"].notna()
    kn = hw[mask_known]
    tw_tvt = tw["TVT"].values; tw_gr = tw["GR"].values
    kn_gr  = kn["GR"].ffill().bfill().values.astype(float)
    kn_tvt = kn["TVT_input"].values
    if len(kn_gr) < 20: continue
    gr_at_tvt = np.interp(kn_tvt, tw_tvt, tw_gr)
    valid = np.isfinite(kn_gr) & np.isfinite(gr_at_tvt)
    if valid.sum() < 20: continue
    c = float(np.corrcoef(kn_gr[valid], gr_at_tvt[valid])[0,1])
    gr_corr_list.append(c)

gr_arr = np.array(gr_corr_list)
print(f"GR vs typewell corr (known zone, 150 wells):")
print(f"  mean : {gr_arr.mean():.3f}")
print(f"  p25  : {np.percentile(gr_arr,25):.3f}")
print(f"  p50  : {np.percentile(gr_arr,50):.3f}")
print(f"  p75  : {np.percentile(gr_arr,75):.3f}")
print(f"  min  : {gr_arr.min():.3f}")
print(f"  >0.8 : {(gr_arr>0.8).sum()}")
print(f"  >0.5 : {(gr_arr>0.5).sum()}")
print(f"  <0.2 : {(gr_arr<0.2).sum()}")

Building dense spatial point cloud...
Dense cloud: 77300 points from 773 wells

TVT RMSE COMPARISON — 200 wells LOO test
Wells tested: 137

Anchor-corrected KNN:
  mean  : 9.7147
  std   : 6.7176
  p50   : 7.7736
  p95   : 22.0527
  max   : 38.2868
  >5ft  : 103
  >10ft : 50

Plain KNN (no anchor):
  mean  : 9.9169
  p50   : 7.6083

Improvement: 2.0%

GR vs TYPEWELL CORRELATION IN KNOWN ZONE
GR vs typewell corr (known zone, 150 wells):
  mean : 0.793
  p25  : 0.749
  p50  : 0.805
  p75  : 0.853
  min  : 0.369
  >0.8 : 83
  >0.5 : 147
  <0.2 : 0


In [8]:
import pandas as pd
import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree
from sklearn.linear_model import Ridge
from sklearn.model_selection import GroupKFold
import warnings
warnings.filterwarnings("ignore")

BASE_DIR  = Path("/kaggle/input/competitions/rogii-wellbore-geology-prediction")
TRAIN_DIR = BASE_DIR / "train"
TEST_DIR  = BASE_DIR / "test"

train_hw  = sorted(TRAIN_DIR.glob("*__horizontal_well.csv"))
train_ids = [f.stem.replace("__horizontal_well","") for f in train_hw]
test_hw   = sorted(TEST_DIR.glob("*__horizontal_well.csv"))
test_ids  = [f.stem.replace("__horizontal_well","") for f in test_hw]
FORMATIONS = ['ANCC','ASTNU','ASTNL','EGFDU','EGFDL','BUDA']

print(f"Train: {len(train_ids)} | Test: {len(test_ids)}")

# ── 1. DENSE SPATIAL CLOUD FROM ALL TRAIN WELLS ──────────────────────────────
print("\nBuilding dense spatial cloud...")
SAMPLE_PER_WELL = 100
rows = []
for wid in train_ids:
    hw  = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    idx = np.linspace(0, len(hw)-1, min(SAMPLE_PER_WELL, len(hw)), dtype=int)
    s   = hw.iloc[idx]
    d   = {"wid": [wid]*len(s),
           "x"  : s["X"].values,
           "y"  : s["Y"].values}
    for f in FORMATIONS:
        d[f] = s[f].values if f in s.columns else np.full(len(s), np.nan)
    rows.append(pd.DataFrame(d))

pts_df   = pd.concat(rows, ignore_index=True)
xy_pts   = pts_df[["x","y"]].values.astype(np.float64)
scale    = xy_pts.std(0); scale[scale < 1] = 1.0
xy_norm  = xy_pts / scale
tree     = cKDTree(xy_norm)
wids_arr = pts_df["wid"].values
print(f"Cloud: {len(pts_df):,} points from {len(train_ids)} wells")

# ── 2. CORE FUNCTIONS ─────────────────────────────────────────────────────────
def knn_form_anchored(ev_xy, anc_xy, anc_f, excl, f_col, k=10):
    """
    Anchor-corrected dense KNN formation top.
    Forces prediction to pass exactly through last-known formation top.
    """
    FETCH = k + 35
    # Predict at anchor point
    da, ia = tree.query((anc_xy/scale).reshape(1,-1), k=FETCH)
    mk = (wids_arr[ia[0]] != excl) if excl else np.ones(FETCH, bool)
    i2 = ia[0][mk][:k]; d2 = da[0][mk][:k]
    if len(i2) == 0: return np.full(len(ev_xy), np.nan)
    fv = pts_df[f_col].iloc[i2].values.astype(float)
    vm = np.isfinite(fv)
    if vm.sum() == 0: return np.full(len(ev_xy), np.nan)
    wa   = 1/(d2[vm]+1e-9); wa /= wa.sum()
    corr = anc_f - float(np.dot(wa, fv[vm]))   # anchor correction

    # Batch predict at all eval points
    de, ie = tree.query(ev_xy/scale, k=FETCH)
    n      = len(ev_xy)
    preds  = np.full(n, np.nan)
    for j in range(n):
        mk_j = (wids_arr[ie[j]] != excl) if excl else np.ones(FETCH, bool)
        ij   = ie[j][mk_j][:k]; dj = de[j][mk_j][:k]
        if len(ij) == 0: continue
        fv_j = pts_df[f_col].iloc[ij].values.astype(float)
        vm_j = np.isfinite(fv_j)
        if vm_j.sum() == 0: continue
        wj   = 1/(dj[vm_j]+1e-9); wj /= wj.sum()
        preds[j] = float(np.dot(wj, fv_j[vm_j])) + corr
    return preds

def gr_global_shift(ev_gr, tvt_init, tw_tvt, tw_gr,
                    search=20, step=0.5):
    """
    Find single global TVT shift that best aligns eval GR to typewell GR.
    Fast O(n × n_search) approach.
    """
    gr = pd.Series(ev_gr).interpolate(limit_direction='both').bfill().ffill().values
    if np.isnan(gr).mean() > 0.6: return 0.0
    tw_s = pd.Series(tw_gr).rolling(5, center=True, min_periods=1).mean().values
    best_sh = 0.0; best_sc = np.inf
    for sh in np.arange(-search, search + step, step):
        gr_tw = np.interp(tvt_init + sh, tw_tvt, tw_s)
        valid = np.isfinite(gr) & np.isfinite(gr_tw)
        if valid.sum() < 10: continue
        sc = float(np.mean((gr[valid] - gr_tw[valid])**2))
        if sc < best_sc: best_sc = sc; best_sh = sh
    return best_sh

def extract_features(hw, tw, wid, is_train):
    """Build complete feature row for eval zone of one well."""
    mask_kn = hw["TVT_input"].notna()
    mask_ev = hw["TVT_input"].isna()
    kn = hw[mask_kn]; ev = hw[mask_ev]
    if len(ev) == 0 or len(kn) < 5: return None

    # Anchor info
    anc_x  = float(kn.iloc[-1]["X"]); anc_y = float(kn.iloc[-1]["Y"])
    anc_xy = np.array([anc_x, anc_y])
    ev_xy  = ev[["X","Y"]].values.astype(np.float64)
    ev_z   = ev["Z"].values; ev_md = ev["MD"].values
    n_ev   = len(ev); excl = wid if is_train else None
    last_tvt = float(kn.iloc[-1]["TVT_input"])
    anc_z  = float(kn.iloc[-1]["Z"]) # Needed for test imputation
    tvt_kn = "TVT" if (is_train and "TVT" in kn.columns) else "TVT_input"
    md_since = ev_md - float(kn.iloc[-1]["MD"])
    frac     = np.arange(n_ev) / max(n_ev-1, 1)

    feats = {
        "frac"    : frac.astype(np.float32),
        "md_since": md_since.astype(np.float32),
    }

    # ── Formation KNN signals (6 formations) ─────────────────────────────────
    tvt_stack = []
    for f in FORMATIONS:
        if is_train:
            if f not in kn.columns: continue
            anc_f = float(kn.iloc[-1][f])
            if not np.isfinite(anc_f): continue
            b_well = float(np.nanmean(kn[tvt_kn].values + kn["Z"].values - kn[f].values))
        else:
            # TEST WELL FIX: Impute anc_f using our spatial tree!
            da, ia = tree.query((anc_xy/scale).reshape(1,-1), k=15)
            fv = pts_df[f].iloc[ia[0]].values.astype(float)
            vm = np.isfinite(fv)
            if vm.sum() == 0: continue
            wa = 1/(da[0][vm]+1e-9); wa /= wa.sum()
            anc_f = float(np.dot(wa, fv[vm]))
            # Force b_well to match the last known TVT exactly
            b_well = last_tvt + anc_z - anc_f

        f_pred = knn_form_anchored(ev_xy, anc_xy, anc_f, excl, f)
        tvt_f  = (-ev_z + f_pred + b_well).astype(np.float32)
        feats[f"tvt_{f}"] = tvt_f
        tvt_stack.append(tvt_f)

    if len(tvt_stack) == 0: return None
    tvt_mat  = np.stack(tvt_stack, 1)
    tvt_mean = np.nanmean(tvt_mat, 1).astype(np.float32)
    tvt_std  = np.nanstd(tvt_mat, 1).astype(np.float32)
    feats["tvt_mean"] = tvt_mean
    feats["tvt_std"]  = tvt_std

    # ── Slope extrapolation from known zone tail ──────────────────────────────
    for tail in [30, 100, 300]:
        kn_t = kn.tail(tail)
        if len(kn_t) < 4: continue
        mdt = kn_t["MD"].values; vt = kn_t["TVT_input"].values
        if np.std(mdt) < 1e-6: continue
        sl = float(np.polyfit(mdt - mdt[-1], vt, 1)[0])
        feats[f"tvt_sl{tail}"] = (last_tvt + sl*md_since).astype(np.float32)

    # ── Z-velocity model: dTVT/dMD = beta*dZ/dMD + intercept ────────────────
    dm = np.diff(kn["MD"].values)
    dz = np.diff(kn["Z"].values)
    dv = np.diff(kn["TVT_input"].values)
    mk = dm > 0
    if mk.sum() >= 10:
        vz = dz[mk]/dm[mk]; vt2 = dv[mk]/dm[mk]
        A  = np.column_stack([vz, np.ones(mk.sum())])
        c, _, _, _ = np.linalg.lstsq(A, vt2, rcond=None)
        ev_dm = np.diff(np.concatenate([[float(kn.iloc[-1]["MD"])], ev_md]))
        ev_dz = np.diff(np.concatenate([[float(kn.iloc[-1]["Z"])],  ev_z]))
        vt_ev = c[0]*(ev_dz/np.where(ev_dm>0,ev_dm,1e-9)) + c[1]
        feats["tvt_vel"] = (last_tvt + np.cumsum(vt_ev*ev_dm)).astype(np.float32)

    # ── GR global shift on mean formation prediction ──────────────────────────
    tw_tvt = tw["TVT"].values; tw_gr = tw["GR"].values
    ev_gr  = ev["GR"].values if "GR" in ev.columns else np.full(n_ev, np.nan)
    shift  = gr_global_shift(ev_gr, tvt_mean, tw_tvt, tw_gr)
    feats["tvt_gr"]   = (tvt_mean + shift).astype(np.float32)
    feats["gr_shift"] = np.full(n_ev, float(shift), np.float32)
    feats["gr_miss"]  = np.full(n_ev, float(np.isnan(ev_gr).mean()), np.float32)

    # ── Last known TVT as reference ───────────────────────────────────────────
    feats["last_tvt"] = np.full(n_ev, last_tvt, np.float32)

    df = pd.DataFrame(feats)
    df["wid"]     = wid
    df["row_idx"] = ev.index.values
    tvt_true = ev["TVT"].values if (is_train and "TVT" in ev.columns) else None
    return df, tvt_true

# ── 3. PROCESS ALL TRAIN WELLS (LOO) ─────────────────────────────────────────
print("\nProcessing train wells (LOO)...")
tr_feats = []; tr_y = []; tr_g = []

for i, wid in enumerate(train_ids):
    if i % 100 == 0: print(f"  {i}/{len(train_ids)}...")
    hw  = pd.read_csv(TRAIN_DIR / f"{wid}__horizontal_well.csv")
    tw  = pd.read_csv(TRAIN_DIR / f"{wid}__typewell.csv").sort_values("TVT")
    res = extract_features(hw, tw, wid, is_train=True)
    if res is None: continue
    feat_df, tvt_true = res
    if tvt_true is None: continue
    tr_feats.append(feat_df.drop(columns=["wid","row_idx"]))
    tr_y.extend(tvt_true.tolist())
    tr_g.extend([wid]*len(feat_df))

X_tr = pd.concat(tr_feats, ignore_index=True).fillna(0).astype(np.float32)
y_tr = np.array(tr_y, dtype=np.float64)
g_tr = np.array(tr_g)
FEAT_COLS = X_tr.columns.tolist()
print(f"X_train: {X_tr.shape}  |  y_train: {y_tr.shape}")
print(f"Features: {FEAT_COLS}")

# ── 4. GROUPKFOLD CV ──────────────────────────────────────────────────────────
print("\n5-Fold GroupKFold CV (group=well)...")
gkf = GroupKFold(n_splits=5)
oof = np.zeros(len(y_tr))
for fold, (ti, vi) in enumerate(gkf.split(X_tr, y_tr, g_tr)):
    m = Ridge(alpha=1.0, positive=True, fit_intercept=True)
    m.fit(X_tr.iloc[ti].values, y_tr[ti])
    oof[vi] = m.predict(X_tr.iloc[vi].values)
    r = float(np.sqrt(np.mean((oof[vi]-y_tr[vi])**2)))
    print(f"  Fold {fold+1}: RMSE = {r:.4f}")

oof_rmse = float(np.sqrt(np.mean((oof - y_tr)**2)))
print(f"\n  OOF RMSE: {oof_rmse:.4f}  ← our CV score")

# ── 5. FINAL MODEL ON ALL TRAIN ───────────────────────────────────────────────
print("\nFitting final Ridge on all train data...")
final_model = Ridge(alpha=1.0, positive=True, fit_intercept=True)
final_model.fit(X_tr.values, y_tr)
print("Feature weights:")
for fn, w in sorted(zip(FEAT_COLS, final_model.coef_), key=lambda x: -x[1]):
    if w > 0.001:
        print(f"  {fn:<20}: {w:.4f}")

# ── 6. PROCESS TEST WELLS ─────────────────────────────────────────────────────
print("\nProcessing test wells...")
te_feats = []; te_ids = []
for wid in test_ids:
    hw  = pd.read_csv(TEST_DIR / f"{wid}__horizontal_well.csv")
    tw  = pd.read_csv(TEST_DIR / f"{wid}__typewell.csv").sort_values("TVT")
    res = extract_features(hw, tw, wid, is_train=False)
    if res is None:
        print(f"  WARNING: {wid} failed"); continue
    feat_df, _ = res
    te_ids.extend([f"{wid}_{idx}" for idx in feat_df["row_idx"].values])
    te_feats.append(feat_df.drop(columns=["wid","row_idx"]))

X_te     = pd.concat(te_feats, ignore_index=True)\
             .reindex(columns=FEAT_COLS, fill_value=0)\
             .fillna(0).astype(np.float32)
te_preds = final_model.predict(X_te.values)
print(f"Test predictions: {len(te_preds)}")
print(f"Range: {te_preds.min():.2f} → {te_preds.max():.2f}")

# ── 7. SUBMISSION ─────────────────────────────────────────────────────────────
print("\nBuilding submission.csv...")
sub      = pd.read_csv(BASE_DIR / "sample_submission.csv")
pred_map = dict(zip(te_ids, te_preds))
sub["tvt"] = sub["id"].map(pred_map)

n_nan = sub["tvt"].isna().sum()
if n_nan > 0:
    print(f"  WARNING: {n_nan} unmatched IDs — filling with median")
    sub["tvt"] = sub["tvt"].fillna(np.nanmedian(te_preds))

sub.to_csv("submission.csv", index=False)
print(f"Saved: submission.csv  shape={sub.shape}")
print(sub.head())
print(f"\nFinal prediction stats:\n{sub['tvt'].describe().round(3)}")
print(f"\nDone. OOF RMSE = {oof_rmse:.4f}")

Train: 773 | Test: 3

Building dense spatial cloud...
Cloud: 77,300 points from 773 wells

Processing train wells (LOO)...
  0/773...
  100/773...
  200/773...
  300/773...
  400/773...
  500/773...
  600/773...
  700/773...
X_train: (3783989, 18)  |  y_train: (3783989,)
Features: ['frac', 'md_since', 'tvt_ANCC', 'tvt_ASTNU', 'tvt_ASTNL', 'tvt_EGFDU', 'tvt_EGFDL', 'tvt_BUDA', 'tvt_mean', 'tvt_std', 'tvt_sl30', 'tvt_sl100', 'tvt_sl300', 'tvt_vel', 'tvt_gr', 'gr_shift', 'gr_miss', 'last_tvt']

5-Fold GroupKFold CV (group=well)...
  Fold 1: RMSE = 22.5965
  Fold 2: RMSE = 26.6334
  Fold 3: RMSE = 14.5981
  Fold 4: RMSE = 22.1218
  Fold 5: RMSE = 16.9437

  OOF RMSE: 21.0230  ← our CV score

Fitting final Ridge on all train data...
Feature weights:
  last_tvt            : 0.9702
  tvt_sl30            : 0.0263
  gr_shift            : 0.0095
  tvt_vel             : 0.0033
  tvt_std             : 0.0031

Processing test wells...
Test predictions: 14151
Range: 11606.03 → 12227.65

Building sub